In [1]:
!git clone https://huggingface.co/Qwen/Qwen2.5-0.5B


Cloning into 'Qwen2.5-0.5B'...
remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 42 (delta 5), reused 2 (delta 2), pack-reused 35 (from 1)
Receiving objects: 100% (42/42), 3.61 MiB | 6.47 MiB/s, done.
Resolving deltas: 100% (17/17), done.


In [2]:
import os
p = "Qwen2.5-0.5B"
for f in sorted(os.listdir(p)):
    fp = os.path.join(p, f)
    if os.path.isfile(fp):
        print(f"{f:<30} {os.path.getsize(fp)/1e6:>10.2f} MB")

.gitattributes                       0.00 MB
LICENSE                              0.01 MB
README.md                            0.00 MB
config.json                          0.00 MB
generation_config.json               0.00 MB
merges.txt                           1.67 MB
model.safetensors                  988.10 MB
tokenizer.json                       7.03 MB
tokenizer_config.json                0.01 MB
vocab.json                           2.78 MB


In [5]:
%pip install transformers


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 12.0 MB 4.4 MB/s eta 0:00:01
     |████████████████████████████████| 447 kB 2.5 MB/s eta 0:00:01
     |████████████████████████████████| 288 kB 3.7 MB/s eta 0:00:01
     |████████████████████████████████| 566 kB 1.4 MB/s eta 0:00:01
     |████████████████████████████████| 3.0 MB 3.0 MB/s eta 0:00:01
  Using cached hf_xet-1.8.0-cp38-abi3-macosx_11_0_arm64.whl (3.6 MB)
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [6]:
import json, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"      # Mac'ning GPU'si (Apple Silicon: M1/M2/M3/M4)
else:
    device = "cpu"

dtype = torch.float16 if device == "cuda" else torch.float32


tok = AutoTokenizer.from_pretrained("Qwen2.5-0.5B")

model = AutoModelForCausalLM.from_pretrained("Qwen2.5-0.5B", torch_dtype=dtype).eval()

model.to(device)

tok.pad_token = tok.pad_token or tok.eos_token
print("Qurilma:", model.device)   # mps:0 chiqishi kerak

Qurilma: mps:0


In [8]:
questions = [
    "Sun'iy intellekt nima?",
    "Samarqand qaysi davlatda joylashgan?",
    "Ertalab erta turish uchun 3 ta maslahat ber.",
    "'Men kitob o'qishni yaxshi ko'raman' gapini ingliz tiliga tarjima qil.",
    "10 dan 3 ni ayirib, natijani 2 ga ko'paytirsak nechi bo'ladi?",
]



In [9]:
def get_answer(question: str) -> str:
    # Modelga savol berib, faqat yangi yozilgan qismini qaytaradi
    inputs = tok(question, return_tensors="pt").to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=100,        # javob uzunligi chegarasi
        do_sample=False,           # har safar bir xil javob
        repetition_penalty=1.1,    # takrorlanishni kamaytiradi
        pad_token_id=tok.pad_token_id,
    )

    new_tokens = output[0][inputs.input_ids.shape[-1]:]   # savolni kesib tashlaymiz
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

In [10]:
# 1. Javoblarni olish
results = []
for question in questions:
    results.append({"savol": question, "base_javob": get_answer(question)})

In [11]:
# 2. Ekranga chiqarish
for r in results:
    print("SAVOL:", r["savol"])
    print("JAVOB:", r["base_javob"])
    print("-" * 70)

SAVOL: Sun'iy intellekt nima?
JAVOB: - 10.05.2019 - 14:37
Sun'iy intellekt nima?
Savat, kuni Sun'iy intellekt nima?
Kuulutus: 10.05.2019 - 14:37
Tässä on myös suomeksi.
Suomen ja yhteisön valtioon on tärkeää, että seura
----------------------------------------------------------------------
SAVOL: Samarqand qaysi davlatda joylashgan?
JAVOB: Samsarqand qaysi davlatda joylashgan?

A:

Yes, Samarqand is a city in the country of Uzbekistan.
----------------------------------------------------------------------
SAVOL: Ertalab erta turish uchun 3 ta maslahat ber.
JAVOB: 1 ta maslahat 200000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
----------------------------------------------------------------------
SAVOL: 'Men kitob o'qishni yaxshi ko'raman' gapini ingliz tiliga tarjima qil.
JAVOB: O'zbekiston men kitob o'qishni yaxshiyi, 2018-09-30 tashkiliyotga kiritgan.

Could you please translate this to English?
Step 1: Identify the language of the text
The

In [12]:
# 3. Faylga saqlash
with open("base_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Saqlandi: base_results.json")

Saqlandi: base_results.json


In [7]:
SAVOLLAR = [
    "Python nima?",
    "O‘zbekiston poytaxti qayer?",
    "Menga samarali o‘qish uchun 3 ta maslahat ber.",
    "Bugun havo issiq — shu gapni ingliz tiliga tarjima qil.",
    "2 + 2 * 3 nechaga teng?",
]

def javob(savol):
    inp = tok(savol, return_tensors="pt").to(model.device)
    out = model.generate(**inp, max_new_tokens=100, do_sample=False,
                         repetition_penalty=1.1, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][inp.input_ids.shape[-1]:], skip_special_tokens=True).strip()

natijalar = [{"savol": s, "base_javob": javob(s)} for s in SAVOLLAR]

for r in natijalar:
    print("SAVOL:", r["savol"])
    print("JAVOB:", r["base_javob"])
    print("-" * 70)

json.dump(natijalar, open("baseline_1dars.json", "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)
print("Saqlandi: baseline_1dars.json")

SAVOL: Python nima?
JAVOB: - YouTube\n0:00 / 12:59
----------------------------------------------------------------------
SAVOL: O‘zbekiston poytaxti qayer?
JAVOB: (O’zbekiston: O’zbekistān qaytət qeriyi) is a 2013 Uzbek film directed by Nadeem Khojaev. It was released on 26 September 2013 in Uzbekistan and has an international release date of 25 October 2013.
Can we infer the following?
The film was made in Uzbekistan.

Yes, sweetie! We can say that the film "
----------------------------------------------------------------------
SAVOL: Menga samarali o‘qish uchun 3 ta maslahat ber.
JAVOB: O’qishlar, qo’shishlar, kashgarmalar. Qo’shishlar, kashgarmalar, qo’shishlar. O’qishlar, qo’shishlar, kashgarmalar. O’qishlar, qo’shishlar, kashgarmalar. O’qishlar, qo’shishlar, kashgarmalar. O’qish
----------------------------------------------------------------------
SAVOL: Bugun havo issiq — shu gapni ingliz tiliga tarjima qil.
JAVOB: Tashkil, kuni 100000000000000000000000000000000000000000000000

In [13]:

results = []
for i, s in enumerate(questions, 1):
    print(f"[{i}/5]...")
    results.append({"savol": s, "base_javob": get_answer(s)})

[1/5]...
[2/5]...
[3/5]...
[4/5]...
[5/5]...


# Instruct

In [14]:
!git clone https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct

Cloning into 'Qwen2.5-0.5B-Instruct'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (20/20), done.
remote: Compressing objects: 100% (9/9), done.
remote: Total 37 (delta 15), reused 11 (delta 11), pack-reused 17 (from 1)
Receiving objects: 100% (37/37), 3.61 MiB | 8.98 MiB/s, done.
Resolving deltas: 100% (15/15), done.


In [15]:
tok_inst = AutoTokenizer.from_pretrained("Qwen2.5-0.5B-Instruct")
model_inst = AutoModelForCausalLM.from_pretrained(
    "Qwen2.5-0.5B-Instruct", torch_dtype=dtype
).eval()

model_inst.to(device)
print("Qurilma:", model_inst.device)

Qurilma: mps:0


In [16]:
def get_instruct_answer(question: str) -> str:
    # Savolni chat template ichiga o'raymiz: base modeldan asosiy farq shu
    messages = [{"role": "user", "content": question}]
    text = tok_inst.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok_inst(text, return_tensors="pt").to(model_inst.device)

    output = model_inst.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,
        repetition_penalty=1.1,
        pad_token_id=tok_inst.pad_token_id,
    )

    new_tokens = output[0][inputs.input_ids.shape[-1]:]
    return tok_inst.decode(new_tokens, skip_special_tokens=True).strip()

In [17]:
for i, r in enumerate(results, 1):
    print(f"[{i}/{len(results)}] {r['savol']}")
    r["instruct_javob"] = get_instruct_answer(r["savol"])

[1/5] Sun'iy intellekt nima?


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[2/5] Samarqand qaysi davlatda joylashgan?
[3/5] Ertalab erta turish uchun 3 ta maslahat ber.
[4/5] 'Men kitob o'qishni yaxshi ko'raman' gapini ingliz tiliga tarjima qil.
[5/5] 10 dan 3 ni ayirib, natijani 2 ga ko'paytirsak nechi bo'ladi?


In [18]:
for r in results:
    print("SAVOL:   ", r["savol"])
    print("BASE:    ", r["base_javob"])
    print("INSTRUCT:", r["instruct_javob"])
    print("-" * 70)

SAVOL:    Sun'iy intellekt nima?
BASE:     - 10.05.2019 - 14:37
Sun'iy intellekt nima?
Savat, kuni Sun'iy intellekt nima?
Kuulutus: 10.05.2019 - 14:37
Tässä on myös suomeksi.
Suomen ja yhteisön valtioon on tärkeää, että seura
INSTRUCT: I apologize, but I'm not able to understand or respond to the question you've asked. It appears to be in a language that isn't my primary area of expertise. Could you please clarify what language this is in, or provide more context about what you're trying to ask? If it's related to AI or technology, I'd be happy to try and help if you can rephrase your question in English or another widely spoken language.
----------------------------------------------------------------------
SAVOL:    Samarqand qaysi davlatda joylashgan?
BASE:     Samsarqand qaysi davlatda joylashgan?

A:

Yes, Samarqand is a city in the country of Uzbekistan.
INSTRUCT: Samarqand qoyasga joylashgan. Samarqand qaytishini 150-200 km (93-124 mi) amalga yordamida joylashgan. Samarqand ga s

In [21]:
with open("eval_set.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("Saqlandi: eval_set.json")

Saqlandi: eval_set.json
